# Learning State-Space Models

In the previous notebook we built a cartpole state-space model and explored how initial-condition errors affect predictions. We assumed our physics model was *correct* — just initialised imperfectly.

Real systems are different. Friction, flex, worn bearings, unmodelled actuator dynamics: the real cartpole behaves differently from our textbook equations even with a perfect initial condition. This is the **model/sim to real gap**, and closing it has arguably been one of the greatest difficulties of deploying AI techniques to robots and other physical systems.

In this notebook we will:
1. Observe the gap by comparing our nominal model to 'real' data with friction.
2. Close it classically — fit the missing friction coefficient $b$ using gradient-based system identification.
3. Close it with a SciML approach — train a residual neural network that learns the *correction* to our physics model.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "../src")

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import trange

from systems.cartpole import CartPoleParams, ObservationNoiseParams
from systems.cartpole import rollout, rollout_with_obs, step

jax.config.update("jax_enable_x64", True)

# --- Shared simulation constants ---
dt = 0.02
duration = 5.0
T = int(duration / dt)
time_arr = jnp.linspace(0, duration, T + 1)

theta0 = jnp.deg2rad(10.0)
x0 = jnp.array([0.0, 0.0, theta0, 0.0])  # [x, x_dot, theta, theta_dot]
actions = jnp.zeros((T, 1))  # zero force (free fall)

key = jax.random.PRNGKey(42)

## The Model/Sim to Real Gap

Our nominal model assumes the cart slides without friction: $b = 0$.

A real cart has viscous friction that acts against its velocity. One way to model it is by introducing a damping term $b\,\dot{x}$ into the equations of motion:
$$\ddot{x} = \frac{F - {\color{red}b\,\dot{x}} + m_p l(\dot{\theta}^2 \sin\theta - \ddot{\theta}\cos\theta)}{M}$$

We will treat the 'real' system as one with $b = 0.5$ N·s/m and pretend we don't know this. Our simulator generates the ground-truth trajectory; a sensor returns noisy partial observations $[x, \theta]$.

In [ ]:
params_nominal = CartPoleParams()  # b=0.0  — what we assume
params_real = CartPoleParams(b=0.5)  # b=0.5  — the "true" system

noise = ObservationNoiseParams()

traj_nominal = rollout(x0, actions, params_nominal, dt=dt)

key, subkey = jax.random.split(key)
traj_real, obs_real = rollout_with_obs(
    x0, actions, params_real, key=subkey, noise=noise, dt=dt
)

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

for ax, (ylabel, state_col, obs_col) in zip(
    axes,
    [
        (r"Cart position $x$ (m)", 0, 0),
        (r"Pole angle $\theta$ (rad)", 2, 1),
    ],
):
    ax.plot(
        time_arr,
        traj_nominal[:, state_col],
        color="#185FA5",
        lw=1.8,
        label="Nominal model ($b=0$)",
    )
    ax.scatter(
        time_arr[::4],
        obs_real[::4, obs_col],
        color="#E05A2B",
        s=10,
        alpha=0.7,
        label="'Real' observations ($b=0.5$)",
    )
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)", fontsize=10)
fig.suptitle("Model-Reality Gap", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    The trajectories diverge even though the initial condition is identical. What does this imply about using a nominal model for long-horizon prediction? How would you go about diagnosing the source of the mismatch in a real experiment?
  </p>
</div>

### Generating a Dataset

Both fitting approaches below minimise the same **weighted multi-step observation-space MSE**:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{k} \sum_{j=1}^{k} \sum_{c \in \{x,\theta\}} w_c \left( \hat{o}_{t+j,c}^{(i)} - o_{t+j,c}^{(i)} \right)^2$$

where $w_c$ is a per-channel weight, $\hat{o}_{t+j,c}$ is the model's predicted observation for channel $c$ at step $j$, and $o_{t+j,c}$ is the corresponding noisy sensor reading.

Weights let us express a preference for accuracy in one channel over another — e.g. penalising position errors more than angle errors. Using multiple future steps improves signal-to-noise ratio: friction has a weak per-step effect but a clearly visible cumulative effect over a longer horizon.

We generate 100 short trajectories from varied initial conditions and split each into non-overlapping $k$-step windows.

In [ ]:
k = 50  # prediction horizon (steps)
N_traj = 100  # number of trajectories to train on
T_train = 100  # trajectory length (steps)
n_windows = T_train // k  # non-overlapping windows per trajectory

actions_k = jnp.zeros((k, 1))
actions_train = jnp.zeros((T_train, 1))


def gen_traj(rng_key):
    k1, k2, k3 = jax.random.split(rng_key, 3)
    theta_i = jax.random.uniform(
        k1, minval=jnp.deg2rad(-20.0), maxval=jnp.deg2rad(20.0)
    )
    x_i = jax.random.uniform(k2, minval=-0.5, maxval=0.5)
    x0_i = jnp.array([x_i, 0.0, theta_i, 0.0])
    return rollout_with_obs(
        x0_i, actions_train, params_real, key=k3, noise=noise, dt=dt
    )


key, subkey = jax.random.split(key)
data_keys = jax.random.split(subkey, N_traj)
states_all, obs_all = jax.vmap(gen_traj)(data_keys)
# states_all: (N_traj, T_train+1, 4)
# obs_all:    (N_traj, T_train+1, 2)

# Split into non-overlapping k-step windows.
# Window i starts at state[i*k] and targets obs[i*k+1 : (i+1)*k+1].
seq_x0s = states_all[:, : n_windows * k : k, :].reshape(-1, 4)
seq_obs = (
    obs_all[:, 1 : n_windows * k + 1, :]
    .reshape(N_traj, n_windows, k, 2)
    .reshape(-1, k, 2)
)

# --- Loss weights: [position x, angle theta] ---
w_x = 10.0
w_theta = 1.0
obs_weights = jnp.array([w_x, w_theta])

print(f"Windows: {seq_x0s.shape[0]}  |  each window: {k} steps × 2 obs channels")
print(f"Loss weights — x: {w_x},  theta: {w_theta}")

## Classical Approach: System Identification

If we more or less know the model structure, and simply have to fit a missing parameter, we can just do that directly. This is a very classical technique known as **system identification**.

The code below leverage JAX's autodiff to find the gradient of the models predictions w.r.t. the unknown parameter $b$, and uses gradient descent to fit it to the data. Note that we are differing from classical system ID methods here in that we are using a variant of stochastic gradient descent (Adam), which is not what's traditionally used for system ID. This is just to keep the code simple and unified with the later SciML approach.

In [ ]:
def sysid_loss(b_scalar):
    params = CartPoleParams(b=b_scalar)

    def single_loss(x0, obs_seq):
        def scan_fn(state, action):
            next_state = step(state, action, params, dt)
            return next_state, next_state

        _, pred_states = jax.lax.scan(scan_fn, x0, actions_k)  # (k, 4)
        pred_obs = pred_states[:, [0, 2]]  # (k, 2)
        return jnp.mean(obs_weights * (pred_obs - obs_seq) ** 2)

    return jnp.mean(jax.vmap(single_loss)(seq_x0s, seq_obs))


b = jnp.array(0.0)
optimizer_sysid = optax.adam(0.05)
opt_state_sysid = optimizer_sysid.init(b)


@jax.jit
def sysid_step(b, opt_state):
    loss, grad = jax.value_and_grad(sysid_loss)(b)
    updates, new_state = optimizer_sysid.update(grad, opt_state)
    new_b = jnp.maximum(optax.apply_updates(b, updates), 0.0)
    return new_b, new_state, loss


sysid_losses = []
pbar = trange(300, desc="Sysid")
for _ in pbar:
    b, opt_state_sysid, loss = sysid_step(b, opt_state_sysid)
    sysid_losses.append(float(loss))
    pbar.set_postfix(loss=f"{float(loss):.2e}", b=f"{float(b):.4f}")

print(f"Fitted  b = {float(b):.4f}")
print("True    b = 0.5000")

In [ ]:
params_fitted = CartPoleParams(b=float(b))
traj_fitted = rollout(x0, actions, params_fitted, dt=dt)

fig = plt.figure(figsize=(12, 5))
gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.35, hspace=0.55)

# Loss curve
ax_loss = fig.add_subplot(gs[:, 0])
ax_loss.semilogy(sysid_losses, color="#1A7A4A", lw=1.5)
ax_loss.set_xlabel("Iteration", fontsize=10)
ax_loss.set_ylabel("Loss (log scale)", fontsize=10)
ax_loss.set_title("Optimisation curve", fontsize=10)
ax_loss.grid(True, alpha=0.3)

# State comparisons
for i, (ylabel, state_col, obs_col) in enumerate(
    [
        (r"$x$ (m)", 0, 0),
        (r"$\theta$ (rad)", 2, 1),
    ]
):
    ax = fig.add_subplot(gs[i, 1])
    ax.plot(
        time_arr,
        traj_nominal[:, state_col],
        color="#185FA5",
        lw=1.5,
        linestyle="--",
        label="Nominal ($b=0$)",
    )
    ax.plot(
        time_arr,
        traj_fitted[:, state_col],
        color="#1A7A4A",
        lw=1.8,
        label=f"Fitted ($b={float(b):.2f}$)",
    )
    ax.scatter(
        time_arr[::4],
        obs_real[::4, obs_col],
        color="#E05A2B",
        s=8,
        alpha=0.6,
        label="Real obs.",
    )
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if i == 1:
        ax.set_xlabel("Time (s)", fontsize=9)

fig.suptitle("System Identification: Fitting Friction", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

<div style="background: #EDFAF3; border-left: 4px solid #1A7A4A; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
    <span style="font-size: 18px;">✏️</span>
    <strong style="color: #1A7A4A; font-size: 15px;">Exercise — Effect of Horizon Length</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    Try changing <code>k</code> to 1 (one-step), 5, and 20. How does the fitted value of $b$ and the convergence speed change? Why does a longer horizon give a better estimate of friction, even though observation noise is the same?
  </p>
</div>

<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; margin-top: 12px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    System identification works great here because we <em>knew</em> friction was the missing physics and we had a parameter slot for it. What happens if the real system has unmodelled effects you haven't anticipated — e.g. nonlinear friction, flex in the pole, or aerodynamic drag? What options do we have? And what if friction/flex/drag change?
  </p>
</div>

## A Hybrid Physics/NN Approach (SciML)

What if the unmodelled physics is complex enough that no simple parametric form captures it? We can instead let a neural network learn the *residual* — the discrepancy between our nominal model and reality:

$$\mathbf{x}_{t+1} = \underbrace{f_\text{physics}(\mathbf{x}_t, \mathbf{a}_t;\, \mathbf{p}_\text{nominal})}_{\text{known physics}} + \underbrace{f_\theta(\mathbf{x}_t, \mathbf{a}_t)}_{\text{learned correction}}$$

This **residual** (or *delta*) model has several advantages over a pure neural approach:
- It only needs to learn the *error*, which is small — easier to fit from limited data.
- At initialisation ($f_\theta \approx 0$), the model falls back to sensible physics.
- Physical structure (energy conservation, symmetries) is preserved by the backbone.

We use the same multi-step observation-space loss as sysid — the network is trained purely on noisy sensor readings $\mathbf{o}_{t+1}, \ldots, \mathbf{o}_{t+k}$ with no access to true states or velocities.

In [ ]:
class ResidualDynamicsModel(eqx.Module):
    """Physics backbone + learned residual correction."""

    net: eqx.nn.MLP
    net_scalar: (
        jax.Array
    )  # Regularization scalar to magnify/dampen the learned correction
    base_params: CartPoleParams = eqx.field(static=True)

    def __init__(self, base_params, key):
        self.base_params = base_params
        self.net_scalar = jnp.array([0.0001, 0.0001, 0.001, 0.001])
        self.net = eqx.nn.MLP(
            in_size=5,  # [x, x_dot, theta, theta_dot, F]
            out_size=4,  # residual correction to full state
            width_size=64,
            depth=3,
            activation=jax.nn.tanh,
            key=key,
        )

    def predict_next(self, state, action):
        physics_next = step(state, action, self.base_params, dt)
        correction = self.net(jnp.concatenate([state, action]))
        return physics_next + self.net_scalar * correction

    def rollout(self, x0, actions):
        def scan_fn(state, action):
            next_state = self.predict_next(state, action)
            return next_state, next_state

        _, states = jax.lax.scan(scan_fn, x0, actions)
        return jnp.concatenate([x0[None], states], axis=0)

In [ ]:
def compute_loss(model, x0s, obs_seqs):
    """Weighted multi-step observation-space MSE over all windows."""

    def single_loss(x0, obs_seq):
        pred_traj = model.rollout(x0, actions_k)  # (k+1, 4)
        pred_obs = pred_traj[1:, [0, 2]]  # (k, 2)
        return jnp.mean(obs_weights * (pred_obs - obs_seq) ** 2)

    return jnp.mean(jax.vmap(single_loss)(x0s, obs_seqs))


key, subkey = jax.random.split(key)
model = ResidualDynamicsModel(params_nominal, key=subkey)

optimizer_nn = optax.adam(2e-4)
opt_state_nn = optimizer_nn.init(eqx.filter(model, eqx.is_array))


@eqx.filter_jit
def nn_train_step(model, x0s, obs_seqs, opt_state):
    loss, grads = eqx.filter_value_and_grad(compute_loss)(model, x0s, obs_seqs)
    updates, new_opt_state = optimizer_nn.update(grads, opt_state)
    new_model = eqx.apply_updates(model, updates)
    return new_model, new_opt_state, loss


nn_losses = []
pbar = trange(1000, desc="NN training")
for _ in pbar:
    model, opt_state_nn, loss = nn_train_step(model, seq_x0s, seq_obs, opt_state_nn)
    nn_losses.append(float(loss))
    pbar.set_postfix(loss=f"{float(loss):.2e}")

In [ ]:
# --- Evaluate on the original held-out trajectory (full 5-second horizon) ---
traj_nn = model.rollout(x0, actions)  # (T+1, 4)

fig = plt.figure(figsize=(12, 5))
gs = gridspec.GridSpec(2, 2, figure=fig, wspace=0.35, hspace=0.55)

ax_loss = fig.add_subplot(gs[:, 0])
ax_loss.semilogy(nn_losses, color="#7B3FA6", lw=1.5)
ax_loss.set_xlabel("Epoch", fontsize=10)
ax_loss.set_ylabel("Loss (log scale)", fontsize=10)
ax_loss.set_title("Training curve", fontsize=10)
ax_loss.grid(True, alpha=0.3)

for i, (ylabel, state_col, obs_col) in enumerate(
    [
        (r"$x$ (m)", 0, 0),
        (r"$\theta$ (rad)", 2, 1),
    ]
):
    ax = fig.add_subplot(gs[i, 1])
    ax.plot(
        time_arr,
        traj_nominal[:, state_col],
        color="#185FA5",
        lw=1.5,
        linestyle="--",
        label="Nominal ($b=0$)",
    )
    ax.plot(
        time_arr, traj_nn[:, state_col], color="#7B3FA6", lw=1.8, label="Residual NN"
    )
    ax.scatter(
        time_arr[::4],
        obs_real[::4, obs_col],
        color="#E05A2B",
        s=8,
        alpha=0.6,
        label="Real obs.",
    )
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if i == 1:
        ax.set_xlabel("Time (s)", fontsize=9)

fig.suptitle(
    "Neural Residual Dynamics: Evaluation on Held-Out Trajectory",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

<div style="background: #EDFAF3; border-left: 4px solid #1A7A4A; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
    <span style="font-size: 18px;">✏️</span>
    <strong style="color: #1A7A4A; font-size: 15px;">Exercise — Structure vs. Flexibility</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    It certainly seems like the SciML approach here performed worse. As a rule of thumb, in the small data limit, models with fewer parameters tend to perform better (unless there are significant transfer learning effects going on). So knowledge about the "true" underlying system can be something that helps learning. What would happen if you completely got rid of the physics backbone in the model above? Try removing it and see what happens.
  </p>
</div>